# **Model Tambahan untuk Tool Calling | MASSIVE-Agents**

---

**Project:** KP Agentic AI + LLM  
**Tahap:** Evaluasi model keempat (Step 7)  
**Model:** LFM2.5-8B-A1B (Liquid AI)  
**Dataset:** MASSIVE-Agents · `en-US` dan `id-ID`

Evaluasi model tambahan menggunakan data, prompt, dan aturan penilaian dari baseline sebelumnya. BFCL AST menjadi metrik utama, sedangkan exact match internal menjadi metrik tambahan. Semua pengujian masih menggunakan data development.

## **1 · Model dan Konfigurasi**

---

LFM2.5-8B-A1B dirancang untuk penggunaan tool calling di perangkat lokal. Model memiliki sekitar 8,3 miliar parameter total dengan sekitar 1,5 miliar parameter aktif. Evaluasi menggunakan varian Ollama Q4_K_M yang ukurannya sekitar 5,2 GB.

Mode `pilot_20` memakai sepuluh ID dalam dua bahasa (20 record). Mode `development_full` memakai 979 record. Kedua mode memakai katalog fungsi dan konfigurasi baseline yang sama.

**Rujukan:** [Liquid AI](https://www.liquid.ai/blog/lfm2-5-8b-a1b) · [Model Ollama](https://ollama.com/library/lfm2.5/tags) · [Model Card](https://huggingface.co/LiquidAI/LFM2.5-8B-A1B)

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import sys
import time
import urllib.error
import urllib.request
from collections import Counter
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
from types import SimpleNamespace

from IPython.display import display

RUN_MODE = "development_full"  # pilot_20 / development_full
MODEL = "lfm2.5:8b-a1b-q4_K_M"
API_BASE = "http://127.0.0.1:11434"
TIMEOUT_SECONDS = 1200
SEED = 20261020
KEEP_ALIVE = "5m"
PROMPT_VERSION = "compact_catalog_v1"
SCORING_VERSION = "internal_exact_match_v1"
BFCL_VERSION = "2025.12.17"
OPTIONS = {
    "temperature": 0,
    "seed": SEED,
    "num_ctx": 16384,
    "num_predict": 384,
    "num_batch": 128,
}
PILOT_IDS = [
    "10093", "11598", "1913", "13907", "1803",
    "2263", "3809", "14601", "5243", "2206",
]

assert RUN_MODE in {"pilot_20", "development_full"}
print("Mode:", RUN_MODE)
print("Model:", MODEL)
print("Python:", sys.version.split()[0])

Mode: development_full
Model: lfm2.5:8b-a1b-q4_K_M
Python: 3.11.9


## **2 · Dataset Development**

---

Data development terdiri dari 582 record bahasa Inggris dan 397 record bahasa Indonesia. Data diambil dari hasil preprocessing yang telah direview. Seluruh record tetap mempertahankan fungsi target dan ground truth asli.

In [2]:
def locate_project():
    current = Path.cwd().resolve()
    for root in [current, *current.parents, Path(r"D:\KP-Agentic-AI")]:
        if (root / "data" / "processed" / "massive_agents_reviewed").is_dir():
            return root
    raise FileNotFoundError("Folder data/processed/massive_agents_reviewed tidak ditemukan.")

ROOT = locate_project()
DATA_DIR = ROOT / "data" / "processed" / "massive_agents_reviewed"
BASELINE_DIR = ROOT / "results" / "baseline_evaluation"
OLD_AST_PATH = ROOT / "results" / "bfcl_ast_evaluation" / "ast_per_prediction.csv"
OUT = ROOT / "results" / "model_tambahan_lfm25"
OUT.mkdir(parents=True, exist_ok=True)


def read_jsonl(path):
    if not path.exists():
        raise FileNotFoundError(path)
    output = []
    with path.open("r", encoding="utf-8-sig") as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            required = {"row_key", "source_record_id", "language_code", "group_id", "split", "question", "function", "gt"}
            if not required.issubset(record):
                raise ValueError(f"Kolom penting tidak lengkap: {record.get('row_key')}")
            if record["split"] != "development" or len(record["function"]) != 55:
                raise ValueError(f"Data atau katalog tidak sesuai: {record['row_key']}")
            if len(record["gt"]) != 1:
                raise ValueError(f"GT tidak tunggal: {record['row_key']}")
            output.append(record)
    return output


def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def write_csv(path, rows, columns=None):
    if not rows and not columns:
        return
    columns = columns or list(rows[0])
    tmp = path.with_suffix(".tmp")
    with tmp.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    tmp.replace(path)

all_rows = []
for language in ("en-US", "id-ID"):
    rows = read_jsonl(DATA_DIR / f"{language}_development.jsonl")
    if len(rows) != (582 if language == "en-US" else 397):
        raise ValueError(f"Jumlah {language} berbeda dari baseline.")
    if any(r["language_code"] != language for r in rows):
        raise ValueError("Label bahasa berbeda.")
    all_rows.extend(rows)

assert len(all_rows) == 979
records_by_key = {r["row_key"]: r for r in all_rows}
assert len(records_by_key) == 979

if RUN_MODE == "pilot_20":
    index = {
        language: {str(r["source_record_id"]): r for r in all_rows if r["language_code"] == language}
        for language in ("en-US", "id-ID")
    }
    selected = []
    for record_id in PILOT_IDS:
        if record_id not in index["en-US"] or record_id not in index["id-ID"]:
            raise ValueError(f"ID pilot tidak ditemukan: {record_id}")
        en, ind = index["en-US"][record_id], index["id-ID"][record_id]
        if next(iter(en["gt"])) != next(iter(ind["gt"])):
            raise ValueError(f"Fungsi target pasangan berbeda: {record_id}")
        selected.extend([en, ind])
    assert len(selected) == 20
else:
    selected = sorted(all_rows, key=lambda r: (str(r["source_record_id"]), r["language_code"]))

assert len(selected) == len({r["row_key"] for r in selected})
print("Folder project:", ROOT)
print("Record development:", len(all_rows))
print("Record pengujian:", len(selected))
print("Bahasa:", dict(Counter(r["language_code"] for r in selected)))

Folder project: D:\KP-Agentic-AI
Record development: 979
Record pengujian: 979
Bahasa: {'en-US': 582, 'id-ID': 397}


## **3 · Prompt Baseline V1**

---

Prompt menampilkan nama fungsi, ringkasan deskripsi, serta nama parameter dari 55 fungsi. Format keluaran berupa satu JSON dengan kunci `name` dan `arguments`. Aturan ini sama dengan baseline tiga model sebelumnya.

Mode utama tetap menggunakan prompt JSON, bukan native tool calling. Dengan begitu, perbandingan model tidak tercampur oleh perbedaan format tugas. Kemampuan native tool calling dapat diuji terpisah sebagai eksperimen tambahan.

In [3]:
def compact_json(value):
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))


def compact_catalog(record):
    lines = []
    for func in record["function"]:
        description = func.get("description", "")
        description = description.split("The default user experience")[0]
        description = description.split(". ")[0].strip()[:100]
        properties = func.get("parameters", {}).get("properties", {})
        arguments = ", ".join(properties.keys())
        lines.append(
            f"{func['name']}: {description}\n"
            f"  Arguments: {arguments if arguments else 'none'}"
        )
    return "\n".join(lines)


def compact_prompt(record):
    return (
        "Select the most appropriate function for the user request.\n"
        "Return one JSON object with exactly two keys: name and arguments.\n"
        "The name must be a function from the catalog.\n"
        "Arguments must contain the relevant parameter names and string values.\n"
        "Do not copy function definitions or include explanations.\n\n"
        "Available functions:\n"
        + compact_catalog(record)
        + "\n\nUser request:\n"
        + record["question"]
        + '\n\nJSON output: {"name":"function_name","arguments":{}}'
    )

# Checksum prompt dan data development penuh dibandingkan konfigurasi baseline asli.
baseline_config = BASELINE_DIR / "config_development_full_v1_verified.json"
if not baseline_config.exists():
    raise FileNotFoundError(f"Konfigurasi baseline belum ditemukan: {baseline_config}")
config_old = json.loads(baseline_config.read_text(encoding="utf-8-sig"))
all_ordered = sorted(all_rows, key=lambda r: (str(r["source_record_id"]), r["language_code"]))

def calc_hashes(rows):
    data_hash = hashlib.sha256()
    prompt_hash = hashlib.sha256()
    for record in rows:
        data_hash.update(compact_json({
            "row_key": record["row_key"], "question": record["question"],
            "function": record["function"], "gt": record["gt"],
        }).encode("utf-8"))
        data_hash.update(b"\n")
        prompt_hash.update(compact_prompt(record).encode("utf-8"))
        prompt_hash.update(b"\n")
    return data_hash.hexdigest(), prompt_hash.hexdigest()

old_data_hash, old_prompt_hash = calc_hashes(all_ordered)
if old_data_hash != config_old["dataset_sha256"] or old_prompt_hash != config_old["prompt_sha256"]:
    raise RuntimeError("Dataset atau prompt tidak identik dengan baseline verified. Hasil tidak boleh dibandingkan langsung.")

selected_data_hash, selected_prompt_hash = calc_hashes(selected)
print("Verifikasi data/prompt baseline: PASS")
print("Hash prompt:", old_prompt_hash[:16], "...")
print("Panjang prompt contoh:", len(compact_prompt(selected[0])))

Verifikasi data/prompt baseline: PASS
Hash prompt: e3d8a54717f244a1 ...
Panjang prompt contoh: 6994


## **4 · Pemeriksaan Model Ollama**

---

Identitas model dan digest Ollama dicatat untuk memastikan hasil dapat ditelusuri. Model tambahan menggunakan tag tetap `lfm2.5:8b-a1b-q4_K_M`.

In [4]:
def ollama_request(endpoint, payload=None, timeout=TIMEOUT_SECONDS):
    body = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = urllib.request.Request(
        API_BASE + endpoint,
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST" if body is not None else "GET",
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:1000]
        raise RuntimeError(f"Ollama HTTP {exc.code}: {detail}") from exc

try:
    tags = ollama_request("/api/tags", timeout=20)
except Exception as exc:
    raise RuntimeError("Ollama lokal belum dapat diakses di http://127.0.0.1:11434.") from exc

installed = {x.get("name"): x for x in tags.get("models", [])}
if MODEL not in installed:
    raise RuntimeError(f"Model {MODEL} belum terpasang. Terminal VS Code: ollama pull {MODEL}")

model_digest = installed[MODEL].get("digest", "")
print("Model:", MODEL)
print("Digest:", model_digest)
print("Ukuran model (GB):", round(installed[MODEL].get("size", 0) / 1e9, 2))

Model: lfm2.5:8b-a1b-q4_K_M
Digest: 9cf756159fc2f3b9128c6a3f544ec90c5e9b8afdbb4179a57b8aea9de589cfb2
Ukuran model (GB): 5.16


## **5 · Penilaian Exact Match Internal**

---

Penilaian internal mempertahankan aturan baseline V1. Pemeriksaan dilakukan pada format JSON, fungsi, nama parameter, serta nilai argumen. Parameter dengan alternatif string kosong diperlakukan sebagai parameter opsional sesuai evaluasi sebelumnya.

In [5]:
def normalize_value(value):
    return " ".join(value.split()).casefold()


def parse_model_json(raw):
    try:
        obj = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return None, "invalid_json"
    if not isinstance(obj, dict):
        return None, "invalid_structure"
    if not isinstance(obj.get("name"), str) or not isinstance(obj.get("arguments"), dict):
        return None, "invalid_structure"
    return obj, "parsed"


def score_internal(prediction, parsing, gt):
    if parsing != "parsed":
        return parsing, False, False
    function_name, gold_args = next(iter(gt.items()))
    if prediction["name"] != function_name:
        return "wrong_function", False, False
    predicted_args = prediction["arguments"]
    gold_keys = set(gold_args)
    if set(predicted_args) - gold_keys:
        return "wrong_argument_keys", True, False
    mandatory = {k for k, alternatives in gold_args.items() if "" not in alternatives}
    if mandatory - set(predicted_args):
        return "wrong_argument_keys", True, False
    for key, value in predicted_args.items():
        allowed = gold_args[key]
        if not isinstance(value, str) or not isinstance(allowed, list):
            return "wrong_argument_values", True, False
        if normalize_value(value) not in {normalize_value(str(item)) for item in allowed}:
            return "wrong_argument_values", True, False
    return "correct", True, True

assert score_internal({"name": "f", "arguments": {}}, "parsed", {"f": {"x": [""]}})[0] == "correct"
assert parse_model_json('{"name":"f","arguments":{}}')[1] == "parsed"
print("Scoring internal: PASS")

Scoring internal: PASS


## **6 · Inferensi Model Tambahan**

---

Prediksi dicatat per record dalam file CSV agar eksperimen dapat dilanjutkan saat terhenti. Hasil model sebelumnya tidak diubah. Kesalahan koneksi dihentikan tanpa menyimpan baris gagal sebagai prediksi baru.

Mode pilot menghasilkan 20 prediksi. Mode development penuh menghasilkan 979 prediksi.

In [6]:
run_tag = RUN_MODE
RESULT_FILE = OUT / f"predictions_{run_tag}.csv"
CONFIG_FILE = OUT / f"config_{run_tag}.json"
AST_FILE = OUT / f"ast_per_prediction_{run_tag}.csv"
SUMMARY_FILE = OUT / f"summary_4_model_{run_tag}.csv"
MCNEMAR_FILE = OUT / f"mcnemar_4_model_{run_tag}.csv"
METADATA_FILE = OUT / f"metadata_{run_tag}.json"

identity = {
    "run_mode": RUN_MODE,
    "model": MODEL,
    "model_digest": model_digest,
    "prompt_version": PROMPT_VERSION,
    "scoring_version": SCORING_VERSION,
    "dataset_sha256": selected_data_hash,
    "prompt_sha256": selected_prompt_hash,
    "record_keys": [r["row_key"] for r in selected],
    "options": OPTIONS,
    "model_transport": "chat_json",
    "keep_alive": KEEP_ALIVE,
    "api_base": API_BASE,
}
fingerprint = hashlib.sha256(compact_json(identity).encode("utf-8")).hexdigest()[:20]
config = {**identity, "fingerprint": fingerprint, "records": len(selected)}

FIELDS = [
    "fingerprint", "model", "row_key", "record_id", "language_code",
    "group_id", "target_function", "prediction_name", "prediction_arguments_json",
    "status", "parsed_json", "function_correct", "full_match", "done_reason",
    "prompt_tokens", "output_tokens", "latency_seconds", "raw_output",
    "error", "timestamp_utc",
]

if CONFIG_FILE.exists():
    previous = json.loads(CONFIG_FILE.read_text(encoding="utf-8-sig"))
    if previous.get("fingerprint") != fingerprint:
        raise RuntimeError("Fingerprint eksperimen berbeda. File hasil lama tidak diubah.")
elif RESULT_FILE.exists():
    raise RuntimeError("File prediksi sudah ada tanpa konfigurasi. Tidak akan ditimpa.")
else:
    CONFIG_FILE.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")

existing = read_csv(RESULT_FILE) if RESULT_FILE.exists() else []
if existing:
    if list(existing[0]) != FIELDS:
        raise RuntimeError("Kolom CSV prediksi tidak sesuai.")
    if any(r["fingerprint"] != fingerprint or r["model"] != MODEL for r in existing):
        raise RuntimeError("Hasil lama berasal dari eksperimen lain.")

completed = {r["row_key"] for r in existing}
if len(completed) != len(existing):
    raise RuntimeError("Ada row_key duplikat di output model baru.")
if not completed.issubset({r["row_key"] for r in selected}):
    raise RuntimeError("Ada row_key di luar subset eksperimen.")

print("Fingerprint:", fingerprint)
print("Prediksi tersimpan:", len(completed))
print("Sisa prediksi:", len(selected) - len(completed))

for number, record in enumerate(selected, 1):
    if record["row_key"] in completed:
        continue
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": compact_prompt(record)}],
        "format": "json",
        "stream": False,
        "options": dict(OPTIONS),
        "keep_alive": KEEP_ALIVE,
    }
    start = time.perf_counter()
    # Kegagalan API tidak dicatat sebagai prediksi agar percobaan dapat diulang.
    response = ollama_request("/api/chat", payload)
    raw = (response.get("message") or {}).get("content") or ""
    prediction, parsing = parse_model_json(raw)
    status, fn_ok, full_ok = score_internal(prediction, parsing, record["gt"])
    if response.get("done_reason") == "length" and status != "correct":
        status = "output_truncated"
    entry = {
        "fingerprint": fingerprint,
        "model": MODEL,
        "row_key": record["row_key"],
        "record_id": record["source_record_id"],
        "language_code": record["language_code"],
        "group_id": record["group_id"],
        "target_function": next(iter(record["gt"])),
        "prediction_name": prediction.get("name", "") if isinstance(prediction, dict) else "",
        "prediction_arguments_json": (compact_json(prediction["arguments"]) if isinstance(prediction, dict) and isinstance(prediction.get("arguments"), dict) else ""),
        "status": status,
        "parsed_json": parsing == "parsed",
        "function_correct": fn_ok,
        "full_match": full_ok,
        "done_reason": response.get("done_reason", ""),
        "prompt_tokens": response.get("prompt_eval_count", ""),
        "output_tokens": response.get("eval_count", ""),
        "latency_seconds": round(time.perf_counter() - start, 2),
        "raw_output": raw,
        "error": "",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    with RESULT_FILE.open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if f.tell() == 0:
            writer.writeheader()
        writer.writerow(entry)
    existing.append(entry)
    completed.add(record["row_key"])
    if number <= 3 or number % 25 == 0 or number == len(selected):
        print(f"{number}/{len(selected)} | {record['row_key']} | {status}")
        if number == 1:
            print("Contoh respons:", raw[:200])

if len(existing) != len(selected):
    raise RuntimeError("Jumlah prediksi belum lengkap.")
print("Inferensi selesai:", len(existing), "prediksi")
print("Kategori:", dict(Counter(r["status"] for r in existing)))

Fingerprint: 2bf989159a08d431f894
Prediksi tersimpan: 0
Sisa prediksi: 979
1/979 | en-US:0001206 | wrong_argument_values
Contoh respons: {
  "name": "cooking.recipe",
  "arguments": {
    "app_name": "default",
    "cooking_type": "general",
    "device_type": "kitchen",
    "food_type": "various",
    "media_type": "text"
  }
}
2/979 | id-ID:0000822 | wrong_argument_values
3/979 | en-US:0001208 | wrong_argument_values
25/979 | id-ID:0000851 | wrong_function
50/979 | en-US:0001283 | wrong_argument_values
75/979 | en-US:0001327 | wrong_function
100/979 | en-US:0001367 | wrong_argument_keys
125/979 | en-US:0001398 | wrong_argument_values
150/979 | en-US:0001459 | wrong_argument_values
175/979 | en-US:0001499 | wrong_argument_values
200/979 | en-US:0001552 | wrong_function
225/979 | en-US:0001607 | wrong_argument_values
250/979 | en-US:0001640 | wrong_function
275/979 | en-US:0001689 | wrong_function
300/979 | en-US:0001727 | wrong_function
325/979 | en-US:0001767 | wrong_function
350/979

## **7 · Evaluasi BFCL AST Resmi**

---

Prediksi dinilai menggunakan fungsi `ast_checker` dari paket `bfcl-eval==2025.12.17`. Skor AST dihitung memakai fungsi dan GT yang sama dengan baseline tiga model. Evaluasi ini menggunakan subset development MASSIVE-Agents, bukan skor leaderboard BFCL penuh.

In [7]:
from importlib.metadata import PackageNotFoundError, version

try:
    installed_bfcl = version("bfcl-eval")
except PackageNotFoundError as exc:
    raise RuntimeError("Kernel ini belum memiliki bfcl-eval. Kernel Python 3.11 (KP BFCL) digunakan pada evaluasi sebelumnya.") from exc
if installed_bfcl != BFCL_VERSION:
    raise RuntimeError(f"Versi BFCL tidak sesuai: {installed_bfcl}; dibutuhkan {BFCL_VERSION}.")

from bfcl_eval.eval_checker.ast_eval.ast_checker import ast_checker
from bfcl_eval.constants.enums import Language
from bfcl_eval.constants.model_config import MODEL_CONFIG_MAPPING

# Model baru didaftarkan hanya pada adapter checker, bukan mengubah evaluasi resmi.
MODEL_CONFIG_MAPPING[MODEL] = SimpleNamespace(underscore_to_dot=False)
MODEL_CONFIG_MAPPING[MODEL.replace("_", "/")] = SimpleNamespace(underscore_to_dot=False)


def as_bool(value):
    return value is True or str(value).strip().lower() == "true"


def score_ast(pred, record):
    if not as_bool(pred["parsed_json"]):
        return False, "adapter:invalid_json", "Output bukan JSON baseline yang valid"
    try:
        args = json.loads(pred["prediction_arguments_json"])
    except (json.JSONDecodeError, TypeError):
        return False, "adapter:invalid_arguments_json", "Argumen tidak valid JSON"
    if not pred["prediction_name"] or not isinstance(args, dict):
        return False, "adapter:invalid_structure", "Nama fungsi atau argumen tidak valid"
    response = ast_checker(
        func_description=record["function"],
        model_output=[{pred["prediction_name"]: args}],
        possible_answer=[record["gt"]],
        language=Language.PYTHON,
        test_category="multiple_function",
        model_name=MODEL,
    )
    if not isinstance(response, dict) or not isinstance(response.get("valid"), bool):
        raise ValueError(f"Respons BFCL AST tidak sesuai: {response!r}")
    return response["valid"], (
        "" if response["valid"] else str(response.get("error_type", "unknown_checker_error"))
    ), json.dumps(response.get("error", []), ensure_ascii=False)

AST_FIELDS = [
    "model", "row_key", "record_id", "language_code", "group_id", "question",
    "target_function", "prediction_function", "prediction_arguments_json",
    "ground_truth_json", "function_correct_internal", "full_match_internal",
    "bfcl_ast_correct", "ast_error_type", "ast_error_message", "previous_status",
]
ast_rows = []
for pred in existing:
    record = records_by_key[pred["row_key"]]
    ok, error_type, error_detail = score_ast(pred, record)
    ast_rows.append({
        "model": MODEL,
        "row_key": pred["row_key"],
        "record_id": pred["record_id"],
        "language_code": pred["language_code"],
        "group_id": pred["group_id"],
        "question": record["question"],
        "target_function": pred["target_function"],
        "prediction_function": pred["prediction_name"],
        "prediction_arguments_json": pred["prediction_arguments_json"],
        "ground_truth_json": json.dumps(record["gt"], ensure_ascii=False),
        "function_correct_internal": as_bool(pred["function_correct"]),
        "full_match_internal": as_bool(pred["full_match"]),
        "bfcl_ast_correct": ok,
        "ast_error_type": error_type,
        "ast_error_message": error_detail,
        "previous_status": pred["status"],
    })

assert len(ast_rows) == len(selected)
write_csv(AST_FILE, ast_rows, AST_FIELDS)
print("BFCL AST benar:", sum(r["bfcl_ast_correct"] for r in ast_rows), "/", len(ast_rows))
print("Exact internal benar:", sum(r["full_match_internal"] for r in ast_rows), "/", len(ast_rows))

BFCL AST benar: 116 / 979
Exact internal benar: 114 / 979


## **8 · Perbandingan Empat Model dan McNemar**

---

Hasil LFM2.5 dibandingkan dengan AST dari Qwen2.5, Gemma 2, dan Sahabat-AI. McNemar menggunakan record yang sama untuk setiap pasangan model. Nilai p dikoreksi dengan metode Holm untuk enam perbandingan dalam masing-masing keluarga uji.

Untuk perbandingan gabungan EN–ID, sejumlah instruksi dapat memiliki hubungan antarbahasa. Hasil per bahasa menjadi pemeriksaan tambahan dan hasil gabungan perlu dibaca dengan hati-hati.

In [8]:
if not OLD_AST_PATH.exists():
    raise FileNotFoundError(f"Hasil AST tiga model belum ditemukan: {OLD_AST_PATH}")
old_ast = read_csv(OLD_AST_PATH)
old_models = list(dict.fromkeys(r["model"] for r in old_ast))
if len(old_models) != 3:
    raise ValueError("AST baseline tidak berisi tepat tiga model.")
expected_all = {r["row_key"] for r in selected}
old_subset = [r for r in old_ast if r["row_key"] in expected_all]
for name in old_models:
    group = [r for r in old_subset if r["model"] == name]
    if len(group) != len(selected) or {r["row_key"] for r in group} != expected_all:
        raise ValueError(f"Hasil AST baseline tidak sejajar untuk: {name}")

all_model_names = old_models + [MODEL]
combined = old_subset + ast_rows
assert len(combined) == 4 * len(selected)
assert len({(r["model"], r["row_key"]) for r in combined}) == len(combined)

summary = []
for name in all_model_names:
    for lang in ("en-US", "id-ID", "ALL"):
        rows = [r for r in combined if r["model"] == name and (lang == "ALL" or r["language_code"] == lang)]
        n = len(rows)
        if not n:
            continue
        a = sum(as_bool(r["bfcl_ast_correct"]) for r in rows)
        b = sum(as_bool(r["full_match_internal"]) for r in rows)
        f = sum(as_bool(r["function_correct_internal"]) for r in rows)
        summary.append({
            "model": name, "language_code": lang, "n": n,
            "ast_correct": a, "ast_accuracy": round(a/n, 6),
            "internal_exact_correct": b, "internal_exact_accuracy": round(b/n, 6),
            "function_correct": f, "function_accuracy": round(f/n, 6),
        })


def mcnemar_pvalue(b, c):
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    log_terms = [
        math.lgamma(n+1) - math.lgamma(i+1) - math.lgamma(n-i+1) - n*math.log(2)
        for i in range(k+1)
    ]
    m = max(log_terms)
    return min(1.0, 2*math.exp(m)*sum(math.exp(x-m) for x in log_terms))

mcnemar_results = []
for metric in ("bfcl_ast_correct", "full_match_internal"):
    for lang in ("ALL", "en-US", "id-ID"):
        score = {
            name: {r["row_key"]: as_bool(r[metric]) for r in combined if r["model"] == name and (lang == "ALL" or r["language_code"] == lang)}
            for name in all_model_names
        }
        keys = set(score[all_model_names[0]])
        assert all(set(score[name]) == keys for name in all_model_names)
        family = []
        for a, b in combinations(all_model_names, 2):
            both_ok = sum(score[a][k] and score[b][k] for k in keys)
            a_only = sum(score[a][k] and not score[b][k] for k in keys)
            b_only = sum(not score[a][k] and score[b][k] for k in keys)
            both_wrong = len(keys) - both_ok - a_only - b_only
            family.append({
                "metric": metric, "language_code": lang,
                "model_a": a, "model_b": b, "n_paired": len(keys),
                "both_correct": both_ok, "a_correct_b_wrong": a_only,
                "a_wrong_b_correct": b_only, "both_wrong": both_wrong,
                "difference_a_minus_b_pp": round(100*(a_only-b_only)/len(keys), 4),
                "p_mcnemar_exact": mcnemar_pvalue(a_only, b_only),
            })
        order = sorted(range(len(family)), key=lambda i: family[i]["p_mcnemar_exact"])
        prev_adj = 0.0
        for rank, i in enumerate(order):
            adjusted = min(1.0, (len(family)-rank) * family[i]["p_mcnemar_exact"])
            prev_adj = max(prev_adj, adjusted)
            family[i]["p_holm"] = prev_adj
            family[i]["significant_0_05"] = prev_adj < 0.05
        mcnemar_results.extend(family)

write_csv(SUMMARY_FILE, summary)
write_csv(MCNEMAR_FILE, mcnemar_results)
try:
    import pandas as pd
    print("Perbandingan AST (semua bahasa):")
    display(pd.DataFrame([r for r in summary if r["language_code"] == "ALL"])[["model", "n", "ast_accuracy", "internal_exact_accuracy", "function_accuracy"]])
    print("McNemar AST + Holm:")
    display(pd.DataFrame([r for r in mcnemar_results if r["metric"] == "bfcl_ast_correct" and r["language_code"] == "ALL"])[["model_a", "model_b", "difference_a_minus_b_pp", "p_holm", "significant_0_05"]])
except ImportError:
    for r in summary:
        if r["language_code"] == "ALL": print(r)
print("Jumlah pasangan McNemar per bahasa:", len(all_model_names)*(len(all_model_names)-1)//2)

Perbandingan AST (semua bahasa):


,model,n,ast_accuracy,internal_exact_accuracy,function_accuracy
0,qwen2.5:7b-instruct,979,0.267620,0.260470,0.792646
1,gemma2:9b-instruct-q4_K_M,979,0.208376,0.205312,0.831461
2,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,979,0.255363,0.250255,0.843718
3,lfm2.5:8b-a1b-q4_K_M,979,0.118488,0.116445,0.454545


McNemar AST + Holm:


,model_a,model_b,difference_a_minus_b_pp,p_holm,significant_0_05
0,qwen2.5:7b-instruct,gemma2:9b-instruct-q4_K_M,5.9244,4.156756e-04,True
1,qwen2.5:7b-instruct,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,1.2257,4.777518e-01,False
2,qwen2.5:7b-instruct,lfm2.5:8b-a1b-q4_K_M,14.9132,1.891218e-19,True
3,gemma2:9b-instruct-q4_K_M,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,-4.6987,6.291359e-04,True
4,gemma2:9b-instruct-q4_K_M,lfm2.5:8b-a1b-q4_K_M,8.9888,2.549677e-08,True
5,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,lfm2.5:8b-a1b-q4_K_M,13.6874,1.151195e-15,True


Jumlah pasangan McNemar per bahasa: 6


## **9 · Metadata dan Output**

---

Seluruh file hasil disimpan terpisah dari baseline lama. Metadata memuat versi checker, digest model, hash prompt, serta jumlah prediksi. Hasil pilot dipisahkan dari hasil development penuh.

In [9]:
metadata = {
    "run": "model_tambahan_lfm25_v1",
    "run_mode": RUN_MODE,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model": MODEL,
    "model_digest": model_digest,
    "dataset_rows": len(selected),
    "prediction_rows": len(existing),
    "prompt_version": PROMPT_VERSION,
    "prompt_sha256": selected_prompt_hash,
    "dataset_sha256": selected_data_hash,
    "baseline_prompt_check": "PASS",
    "options": OPTIONS,
    "model_transport": "chat_json",
    "bfcl_eval_version": installed_bfcl,
    "checker": "bfcl_eval.eval_checker.ast_eval.ast_checker.ast_checker",
    "split": "development",
    "test_set_used": False,
    "comparison_models": all_model_names,
    "notes": [
        "Output 55 fungsi menggunakan prompt JSON yang sama dengan baseline.",
        "AST resmi dievaluasi pada subset development MASSIVE-Agents, bukan leaderboard BFCL penuh.",
        "McNemar gabungan perlu memperhatikan pasangan record EN-ID.",
        "Audit GT yang belum final tetap bersifat provisional.",
        "Varian model dan konfigurasi tidak diubah sepanjang satu eksperimen.",
    ],
}
METADATA_FILE.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
for path in (RESULT_FILE, CONFIG_FILE, AST_FILE, SUMMARY_FILE, MCNEMAR_FILE, METADATA_FILE):
    print(path.relative_to(ROOT), "|", path.stat().st_size, "bytes")
print("Status: seluruh hasil", RUN_MODE, "tersimpan")

results\model_tambahan_lfm25\predictions_development_full.csv | 390729 bytes
results\model_tambahan_lfm25\config_development_full.json | 22291 bytes
results\model_tambahan_lfm25\ast_per_prediction_development_full.csv | 477092 bytes
results\model_tambahan_lfm25\summary_4_model_development_full.csv | 1067 bytes
results\model_tambahan_lfm25\mcnemar_4_model_development_full.csv | 5833 bytes
results\model_tambahan_lfm25\metadata_development_full.json | 1479 bytes
Status: seluruh hasil development_full tersimpan
